In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import export_graphviz
import pydotplus
import os
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Image
from deap import base, creator, tools, algorithms
import random
from imblearn.over_sampling import SMOTE
from collections import Counter

### Preprocessing the non-ensemble data. 

# Comment me out

In [2]:
# import pandas as pd

# # Load the Excel file
# file_path = 'uneditedDUMMY-Modified Combined Waveform Output-NonEnsemble.xlsx'
# df = pd.read_excel(file_path)

# # Initialize a dictionary to keep track of the counts of each entry
# entry_counts = {}

# # Function to modify the "Parent Folder" entry with a number for repeats
# def modify_parent_folder(entry):
#     if entry in entry_counts:
#         entry_counts[entry] += 1
#     else:
#         entry_counts[entry] = 1
#     return f"{entry}_{entry_counts[entry]}"

# # Apply the function to the "Parent Folder" column
# df['Parent Folder'] = df['Parent Folder'].apply(modify_parent_folder)

# # Save the modified dataframe to a new Excel file (optional)
# output_path = 'REAL-Modified Combined Waveform Output-NonEnsemble.xlsx'
# df.to_excel(output_path, index=False)

# print(f"Modified file saved as {output_path}")

In [3]:
import pandas as pd

# Load the Excel file
file_path = 'DUMMY-Modified Combined Waveform Output-NonEnsemble.xlsx'
df = pd.read_excel(file_path)

# Combine "Parent Folder" and "Subject ID" into a new "Parent Folder" column
df['Parent Folder'] = df['Parent Folder'] + '_' + df['Subject ID']

# Drop the original "Subject ID" column
df = df.drop(columns=['Subject ID'])

# Save the modified dataframe to a new Excel file (optional)
output_path = 'REAL-Modified Combined Waveform Output-NonEnsemble.xlsx'
df.to_excel(output_path, index=False)

print(f"Modified file saved as {output_path}")


Modified file saved as REAL-Modified Combined Waveform Output-NonEnsemble.xlsx


# Process the Excel File to make it friendly to this code

In [4]:
# Step 1: Load the data from Excel
# file_path = "Modified Combined Waveform Output-Normal.xlsx"
# file_path = "Modified Combined Waveform Output-Enhanced.xlsx"
# file_path='DUMMY-Modified Combined Waveform Output-NonEnsemble.xlsx'
file_path='REAL-Modified Combined Waveform Output-NonEnsemble.xlsx'

df = pd.read_excel(file_path)

In [5]:
# Step 2: Print the column names and the first few rows of the DataFrame to verify content
print("Column names in the DataFrame:")
print(df.columns)

Column names in the DataFrame:
Index(['Parent Folder', 'Hemorrhage Level', 'Occlusion Group', 'Avg', 'Max',
       'Min', 'Title', 'Time Point', 'Heart Rate', 'Integral', 'Derivative',
       'Sign Changes', 'Original Data', 'Pulse Pressure',
       'Augmentation Pressure', 'Pseudo-Map', 'Inflection Time',
       'FFT Frequencies', 'FFT Power Spectral Density', 'Filtered Data',
       'Highest PSD Value 1', 'Frequency 1', 'Highest PSD Value 2',
       'Frequency 2', 'Highest PSD Value 3', 'Frequency 3',
       'Highest PSD Value 4', 'Frequency 4', 'Highest PSD Value 5',
       'Frequency 5', 'Peak-to-Peak Interval', 'Heart Rate Interval',
       'Time to Systolic Rise', 'Time to Systolic Decline',
       'Time to Deceleration', 'Time to Systolic Peak',
       'Time to Diastolic Peak', 'Average Systolic Pressure',
       'Average Diastolic Pressure', 'Average Systolic Rise Pressure',
       'Average Systolic Decline Pressure', 'Average Deceleration Pressure',
       'Average Systolic Pr

In [6]:
print("\nFirst few rows of the DataFrame:")
df.head()


First few rows of the DataFrame:


,Parent Folder,Hemorrhage Level,Occlusion Group,Avg,Max,Min,Title,Time Point,Heart Rate,Integral,...,Deceleration Area (No Diastolic),"Deceleration Area (Normalized, No Diastolic)",Diastolic Area (No Diastolic),"Diastolic Area (Normalized, No Diastolic)",Slope of Diastolic Pressure,Slope of Systolic Pressure,Slope of Descending Systolic Pressure,Time,Location,Flow/Pressure/Vol
0,ERNE5_Sample1,30,NaN,66.818785,82.560872,43.290238,T0-30% No Occlusion Proximal Pressure,0,137.931034,5765.499394,...,2552.222598,29.335892,2552.222598,29.335892,-0.390343,0.606150,-0.880411,T0-30%,No Occlusion Proximal,Pressure
1,ERNE5_Sample2,30,NaN,67.170976,83.943414,47.357817,T0-30% No Occlusion Proximal Pressure,0,134.831461,5928.331480,...,2638.506097,29.646136,2638.506097,29.646136,-0.390301,0.514059,-0.997643,T0-30%,No Occlusion Proximal,Pressure
2,ERNE5_Sample3,30,NaN,68.591390,85.506391,43.892746,T0-30% No Occlusion Proximal Pressure,0,133.333333,6124.401374,...,2686.630146,29.851446,2686.630146,29.851446,-0.395623,0.614779,-0.871788,T0-30%,No Occlusion Proximal,Pressure
3,ERNE5_Sample4,30,NaN,70.640432,87.798026,41.287680,T0-30% No Occlusion Proximal Pressure,0,136.363636,6168.346137,...,2675.598835,30.404532,2675.598835,30.404532,-0.420114,0.725761,-0.911053,T0-30%,No Occlusion Proximal,Pressure
4,ERNE5_Sample5,30,NaN,70.759980,87.260818,43.406019,T0-30% No Occlusion Proximal Pressure,0,137.931034,6107.104402,...,2694.565786,30.972021,2694.565786,30.972021,-0.437276,0.708273,-0.893270,T0-30%,No Occlusion Proximal,Pressure


In [7]:
df.shape

(33980, 74)

In [8]:
# Step 3: Strip whitespace from column names
df.columns = df.columns.str.strip()

# Step 4: Ensure that the 'Hemorrhage Level' column exists
if 'Hemorrhage Level' not in df.columns:
    raise KeyError("'Hemorrhage Level' column is missing from the DataFrame.")

In [9]:
# Step 5: Filter rows where 'Flow/Pressure/Vol' is 'Pressure'
df = df[df['Flow/Pressure/Vol'].str.contains('Pressure', na=False)]
print("\nData after filtering 'Flow/Pressure/Vol':")
df.head()


Data after filtering 'Flow/Pressure/Vol':


,Parent Folder,Hemorrhage Level,Occlusion Group,Avg,Max,Min,Title,Time Point,Heart Rate,Integral,...,Deceleration Area (No Diastolic),"Deceleration Area (Normalized, No Diastolic)",Diastolic Area (No Diastolic),"Diastolic Area (Normalized, No Diastolic)",Slope of Diastolic Pressure,Slope of Systolic Pressure,Slope of Descending Systolic Pressure,Time,Location,Flow/Pressure/Vol
0,ERNE5_Sample1,30,NaN,66.818785,82.560872,43.290238,T0-30% No Occlusion Proximal Pressure,0,137.931034,5765.499394,...,2552.222598,29.335892,2552.222598,29.335892,-0.390343,0.606150,-0.880411,T0-30%,No Occlusion Proximal,Pressure
1,ERNE5_Sample2,30,NaN,67.170976,83.943414,47.357817,T0-30% No Occlusion Proximal Pressure,0,134.831461,5928.331480,...,2638.506097,29.646136,2638.506097,29.646136,-0.390301,0.514059,-0.997643,T0-30%,No Occlusion Proximal,Pressure
2,ERNE5_Sample3,30,NaN,68.591390,85.506391,43.892746,T0-30% No Occlusion Proximal Pressure,0,133.333333,6124.401374,...,2686.630146,29.851446,2686.630146,29.851446,-0.395623,0.614779,-0.871788,T0-30%,No Occlusion Proximal,Pressure
3,ERNE5_Sample4,30,NaN,70.640432,87.798026,41.287680,T0-30% No Occlusion Proximal Pressure,0,136.363636,6168.346137,...,2675.598835,30.404532,2675.598835,30.404532,-0.420114,0.725761,-0.911053,T0-30%,No Occlusion Proximal,Pressure
4,ERNE5_Sample5,30,NaN,70.759980,87.260818,43.406019,T0-30% No Occlusion Proximal Pressure,0,137.931034,6107.104402,...,2694.565786,30.972021,2694.565786,30.972021,-0.437276,0.708273,-0.893270,T0-30%,No Occlusion Proximal,Pressure


In [10]:
# Step 6: Define the list of allowed locations and filter rows based on 'Location'
allowed_locations_keywords = ['Proximal', 'Distal']
df = df[df['Location'].str.contains('|'.join(allowed_locations_keywords), na=False)]
df.head()

,Parent Folder,Hemorrhage Level,Occlusion Group,Avg,Max,Min,Title,Time Point,Heart Rate,Integral,...,Deceleration Area (No Diastolic),"Deceleration Area (Normalized, No Diastolic)",Diastolic Area (No Diastolic),"Diastolic Area (Normalized, No Diastolic)",Slope of Diastolic Pressure,Slope of Systolic Pressure,Slope of Descending Systolic Pressure,Time,Location,Flow/Pressure/Vol
0,ERNE5_Sample1,30,NaN,66.818785,82.560872,43.290238,T0-30% No Occlusion Proximal Pressure,0,137.931034,5765.499394,...,2552.222598,29.335892,2552.222598,29.335892,-0.390343,0.606150,-0.880411,T0-30%,No Occlusion Proximal,Pressure
1,ERNE5_Sample2,30,NaN,67.170976,83.943414,47.357817,T0-30% No Occlusion Proximal Pressure,0,134.831461,5928.331480,...,2638.506097,29.646136,2638.506097,29.646136,-0.390301,0.514059,-0.997643,T0-30%,No Occlusion Proximal,Pressure
2,ERNE5_Sample3,30,NaN,68.591390,85.506391,43.892746,T0-30% No Occlusion Proximal Pressure,0,133.333333,6124.401374,...,2686.630146,29.851446,2686.630146,29.851446,-0.395623,0.614779,-0.871788,T0-30%,No Occlusion Proximal,Pressure
3,ERNE5_Sample4,30,NaN,70.640432,87.798026,41.287680,T0-30% No Occlusion Proximal Pressure,0,136.363636,6168.346137,...,2675.598835,30.404532,2675.598835,30.404532,-0.420114,0.725761,-0.911053,T0-30%,No Occlusion Proximal,Pressure
4,ERNE5_Sample5,30,NaN,70.759980,87.260818,43.406019,T0-30% No Occlusion Proximal Pressure,0,137.931034,6107.104402,...,2694.565786,30.972021,2694.565786,30.972021,-0.437276,0.708273,-0.893270,T0-30%,No Occlusion Proximal,Pressure


In [11]:
df.shape

(33980, 74)

In [12]:
# Step 6: Define the list of allowed locations and filter rows based on 'Location'
allowed_locations_keywords = ['Proximal','Distal']

# Create a new column 'Location_Matched' to indicate which keyword was matched
def match_location(row):
    for keyword in allowed_locations_keywords:
        if keyword in str(row['Location']):
            return keyword
    return np.nan

# Apply the function to create the 'Location_Matched' column
df['Location_Matched'] = df.apply(match_location, axis=1)

# Filter rows where 'Location_Matched' is not NaN
df = df.dropna(subset=['Location_Matched'])
print("\nData after filtering 'Location' and adding 'Location_Matched':")
print(df.head())



Data after filtering 'Location' and adding 'Location_Matched':
   Parent Folder  Hemorrhage Level  Occlusion Group        Avg        Max  \
0  ERNE5_Sample1                30              NaN  66.818785  82.560872   
1  ERNE5_Sample2                30              NaN  67.170976  83.943414   
2  ERNE5_Sample3                30              NaN  68.591390  85.506391   
3  ERNE5_Sample4                30              NaN  70.640432  87.798026   
4  ERNE5_Sample5                30              NaN  70.759980  87.260818   

         Min                                  Title  Time Point  Heart Rate  \
0  43.290238  T0-30% No Occlusion Proximal Pressure           0  137.931034   
1  47.357817  T0-30% No Occlusion Proximal Pressure           0  134.831461   
2  43.892746  T0-30% No Occlusion Proximal Pressure           0  133.333333   
3  41.287680  T0-30% No Occlusion Proximal Pressure           0  136.363636   
4  43.406019  T0-30% No Occlusion Proximal Pressure           0  137.931034   

In [13]:
# Step 7: Modify Hemorrhage Level based on Time Point
def modify_hemorrhage_level(row):
    time_point = row['Time Point']
    hemorrhage_level = row['Hemorrhage Level']
    
    if time_point == 0:
        return 0
    elif time_point in [25, 26, 30]:
#         return hemorrhage_level
        return float(hemorrhage_level)
#     else:
#         adjusted_level = (time_point / 30) * hemorrhage_level
#         return np.floor(adjusted_level / 10) * 10

# Apply the function to the 'Hemorrhage Level' column
df['Hemorrhage Level'] = df.apply(modify_hemorrhage_level, axis=1)
print("\nData after modifying 'Hemorrhage Level':")
print(df.head())   


Data after modifying 'Hemorrhage Level':
   Parent Folder  Hemorrhage Level  Occlusion Group        Avg        Max  \
0  ERNE5_Sample1               0.0              NaN  66.818785  82.560872   
1  ERNE5_Sample2               0.0              NaN  67.170976  83.943414   
2  ERNE5_Sample3               0.0              NaN  68.591390  85.506391   
3  ERNE5_Sample4               0.0              NaN  70.640432  87.798026   
4  ERNE5_Sample5               0.0              NaN  70.759980  87.260818   

         Min                                  Title  Time Point  Heart Rate  \
0  43.290238  T0-30% No Occlusion Proximal Pressure           0  137.931034   
1  47.357817  T0-30% No Occlusion Proximal Pressure           0  134.831461   
2  43.892746  T0-30% No Occlusion Proximal Pressure           0  133.333333   
3  41.287680  T0-30% No Occlusion Proximal Pressure           0  136.363636   
4  43.406019  T0-30% No Occlusion Proximal Pressure           0  137.931034   

      Integral  ... 

In [14]:
# Step 8: Define the ground truth labels
df['Hemorrhage Level'] = df['Hemorrhage Level'].astype(str)

# Step 9: Combine Parent Folder and Time Point to uniquely identify each patient
df['Patient ID'] = df['Parent Folder'].astype(str) + "_" + df['Time Point'].astype(str)

In [15]:
df.head()

,Parent Folder,Hemorrhage Level,Occlusion Group,Avg,Max,Min,Title,Time Point,Heart Rate,Integral,...,Diastolic Area (No Diastolic),"Diastolic Area (Normalized, No Diastolic)",Slope of Diastolic Pressure,Slope of Systolic Pressure,Slope of Descending Systolic Pressure,Time,Location,Flow/Pressure/Vol,Location_Matched,Patient ID
0,ERNE5_Sample1,0.0,NaN,66.818785,82.560872,43.290238,T0-30% No Occlusion Proximal Pressure,0,137.931034,5765.499394,...,2552.222598,29.335892,-0.390343,0.606150,-0.880411,T0-30%,No Occlusion Proximal,Pressure,Proximal,ERNE5_Sample1_0
1,ERNE5_Sample2,0.0,NaN,67.170976,83.943414,47.357817,T0-30% No Occlusion Proximal Pressure,0,134.831461,5928.331480,...,2638.506097,29.646136,-0.390301,0.514059,-0.997643,T0-30%,No Occlusion Proximal,Pressure,Proximal,ERNE5_Sample2_0
2,ERNE5_Sample3,0.0,NaN,68.591390,85.506391,43.892746,T0-30% No Occlusion Proximal Pressure,0,133.333333,6124.401374,...,2686.630146,29.851446,-0.395623,0.614779,-0.871788,T0-30%,No Occlusion Proximal,Pressure,Proximal,ERNE5_Sample3_0
3,ERNE5_Sample4,0.0,NaN,70.640432,87.798026,41.287680,T0-30% No Occlusion Proximal Pressure,0,136.363636,6168.346137,...,2675.598835,30.404532,-0.420114,0.725761,-0.911053,T0-30%,No Occlusion Proximal,Pressure,Proximal,ERNE5_Sample4_0
4,ERNE5_Sample5,0.0,NaN,70.759980,87.260818,43.406019,T0-30% No Occlusion Proximal Pressure,0,137.931034,6107.104402,...,2694.565786,30.972021,-0.437276,0.708273,-0.893270,T0-30%,No Occlusion Proximal,Pressure,Proximal,ERNE5_Sample5_0


In [16]:
# Step 10: Drop the specified columns
columns_to_drop = [
    'Occlusion Group', 'Integral', 'Derivative', 'Sign Changes', 'Original Data',
    'FFT Frequencies', 'FFT Power Spectral Density', 'Filtered Data', 'Time', 
    'Flow/Pressure/Vol',
    'Parent Folder', 'Title', 'Time Point','Location'
]
df = df.drop(columns=[col for col in columns_to_drop if col in df.columns], errors='ignore')

# Drop any columns that are completely empty
df = df.dropna(axis=1, how='all')
print("\nData after dropping specified columns and empty columns:")
print(df.head())


Data after dropping specified columns and empty columns:
  Hemorrhage Level        Avg        Max        Min  Heart Rate  \
0              0.0  66.818785  82.560872  43.290238  137.931034   
1              0.0  67.170976  83.943414  47.357817  134.831461   
2              0.0  68.591390  85.506391  43.892746  133.333333   
3              0.0  70.640432  87.798026  41.287680  136.363636   
4              0.0  70.759980  87.260818  43.406019  137.931034   

   Pulse Pressure  Augmentation Pressure  Pseudo-Map  Inflection Time  \
0       39.270634             -13.206169   56.380449            0.215   
1       36.585597             -13.966997   59.553016            0.220   
2       41.613645             -13.948604   57.763961            0.225   
3       46.510346             -14.576851   56.791128            0.220   
4       43.854799             -13.399056   58.024285            0.215   

   Highest PSD Value 1  ...  Systolic Rise Area (Normalized, No Diastolic)  \
0          1144.431627

In [17]:
# Step 11: Split the DataFrame based on 'Location_Matched' and update column headers
dfs = {}
for location in df['Location_Matched'].unique():
    df_location = df[df['Location_Matched'] == location].copy()
    # Rename columns to include location
    df_location.columns = [f"{col} {location}" if col not in ['Patient ID', 'Location_Matched'] else col for col in df_location.columns]
    dfs[location] = df_location

# Combine the split DataFrames based on 'Patient ID'
# Start with an empty DataFrame to combine results
df_combined = pd.DataFrame()

# List to keep track of columns added to avoid duplicates
used_columns = set()

for location, df_location in dfs.items():
    if df_combined.empty:
        df_combined = df_location
    else:
        # Merge DataFrames based on 'Patient ID'
        df_combined = pd.merge(df_combined, df_location, on='Patient ID', how='outer')

# Drop duplicate columns if any, keeping the first occurrence
df_combined = df_combined.loc[:, ~df_combined.columns.duplicated()]

In [18]:

# Step 12: Keep only one column for 'Hemorrhage Level' and remove duplicates
hemorrhage_level_columns = [col for col in df_combined.columns if 'Hemorrhage Level' in col]
if hemorrhage_level_columns:
    # Create a new column for 'Hemorrhage Level' and keep only the first occurrence
    df_combined['Hemorrhage Level'] = df_combined[hemorrhage_level_columns[0]]
    # Drop all original 'Hemorrhage Level' columns, including the first one
    df_combined = df_combined.drop(columns=hemorrhage_level_columns)

# Remove columns containing the phrase 'Location_Matched'
df_combined = df_combined.loc[:, ~df_combined.columns.str.contains('Location_Matched')]

In [19]:
df_combined.head()

,Avg Proximal,Max Proximal,Min Proximal,Heart Rate Proximal,Pulse Pressure Proximal,Augmentation Pressure Proximal,Pseudo-Map Proximal,Inflection Time Proximal,Highest PSD Value 1 Proximal,Frequency 1 Proximal,...,Systolic Rise Area (No Diastolic) Distal,"Systolic Rise Area (Normalized, No Diastolic) Distal",Deceleration Area (No Diastolic) Distal,"Deceleration Area (Normalized, No Diastolic) Distal",Diastolic Area (No Diastolic) Distal,"Diastolic Area (Normalized, No Diastolic) Distal",Slope of Diastolic Pressure Distal,Slope of Systolic Pressure Distal,Slope of Descending Systolic Pressure Distal,Hemorrhage Level
0,66.818785,82.560872,43.290238,137.931034,39.270634,-13.206169,56.380449,0.215,1144.431627,12.695312,...,1674.330915,19.245183,2079.698640,23.904582,2079.698640,23.904582,-0.170179,0.164441,-0.061214,0.0
1,67.170976,83.943414,47.357817,134.831461,36.585597,-13.966997,59.553016,0.220,1928.606342,12.695312,...,1725.833319,19.391386,2157.949524,24.246624,2157.949524,24.246624,-0.176870,0.168624,-0.034764,0.0
2,68.591390,85.506391,43.892746,133.333333,41.613645,-13.948604,57.763961,0.225,1399.771230,12.695312,...,2338.181450,25.979794,2239.483178,24.883146,2239.483178,24.883146,-0.192483,0.186974,0.000000,0.0
3,70.640432,87.798026,41.287680,136.363636,46.510346,-14.576851,56.791128,0.220,1651.176101,9.765625,...,1633.599960,18.563636,2272.325472,25.821880,2272.325472,25.821880,-0.195091,0.226287,-0.004537,0.0
4,70.759980,87.260818,43.406019,137.931034,43.854799,-13.399056,58.024285,0.215,1694.358679,45.898438,...,1643.542798,18.891297,2224.251527,25.566110,2224.251527,25.566110,-0.196811,0.207874,-0.013645,0.0


In [20]:
print(df_combined.shape)

(16990, 120)


In [21]:
# # Optional: Save the cleaned data to a new Excel file
# df=df_combined
# # output_file_path = "Random Forest Input.xlsx"

# # Determine the output file path based on the input file path
# if file_path == "Modified Combined Waveform Output-Normal.xlsx":
#     output_file_path = "Random Forest Input.xlsx"
# elif file_path == "Modified Combined Waveform Output-Enhanced.xlsx":
#     output_file_path = "Enhanced Random Forest Input.xlsx"
# elif file_path == "REAL-Modified Combined Waveform Output-NonEnsemble.xlsx":
#     output_file_path = "NonEnsemble-Enhanced Random Forest Input.xlsx"

# #     output_file_path='REAL-Modified Combined Waveform Output-NonEnsemble.xlsx'
# else:
#     output_file_path = "RF Input.xlsx"  # Handle the case where file_path does not match any known values

# print(f"The output file path is '{output_file_path}'")

# df.to_excel(output_file_path, index=False)

# print("\nData processing complete. The new file has been saved as 'Random Forest Input.xlsx'.")

In [22]:
# import pandas as pd
# import xlsxwriter

# # Assign the combined DataFrame
# df = df_combined

# # Determine the output file path based on the input file path
# if file_path == "Modified Combined Waveform Output-Normal.xlsx":
#     output_file_path = "Random Forest Input.xlsx"
# elif file_path == "Modified Combined Waveform Output-Enhanced.xlsx":
#     output_file_path = "Enhanced Random Forest Input.xlsx"
# elif file_path == "REAL-Modified Combined Waveform Output-NonEnsemble.xlsx":
#     output_file_path = "NonEnsemble-Enhanced Random Forest Input.xlsx"
# elif file_path == "DUMMY-Modified Combined Waveform Output-NonEnsemble.xlsx":
#     output_file_path = "NonEnsemble-Enhanced Random Forest Input.xlsx"
# else:
#     output_file_path = "RF Input.xlsx"  # Handle the case where file_path does not match any known values

# print(f"The output file path is '{output_file_path}'")

# # Maximum rows allowed per sheet
# max_rows_per_sheet = 17179869184

# # Initialize the Excel writer with xlsxwriter and enable ZIP64
# with pd.ExcelWriter(output_file_path, engine='xlsxwriter') as writer:
#     writer.book.use_zip64()
    
#     # Split DataFrame into chunks and write each chunk to a separate sheet
#     for i in range(0, len(df), max_rows_per_sheet):
#         chunk = df.iloc[i:i + max_rows_per_sheet]
#         sheet_name = f'Sheet_{i // max_rows_per_sheet + 1}'
#         chunk.to_excel(writer, sheet_name=sheet_name, index=False)

# print(f"Data processing complete. The new file has been saved as '{output_file_path}'.")


In [23]:
import pandas as pd
import xlsxwriter

# Assign the combined DataFrame
df = df_combined

# Determine the output file path based on the input file path
if file_path == "Modified Combined Waveform Output-Normal.xlsx":
    output_file_path = "Random Forest Input.xlsx"
elif file_path == "Modified Combined Waveform Output-Enhanced.xlsx":
    output_file_path = "Enhanced Random Forest Input.xlsx"
elif file_path == "REAL-Modified Combined Waveform Output-NonEnsemble.xlsx":
    output_file_path = "NonEnsemble-Enhanced Random Forest Input.xlsx"
elif file_path == "DUMMY-Modified Combined Waveform Output-NonEnsemble.xlsx":
    output_file_path = "NonEnsemble-Enhanced Random Forest Input.xlsx"
else:
    output_file_path = "RF Input.xlsx"  # Handle the case where file_path does not match any known values

print(f"The output file path is '{output_file_path}'")

# Maximum rows allowed per sheet (Excel's limit is 1,048,576 rows per sheet)
max_rows_per_sheet = 1048576

# Initialize the Excel writer with xlsxwriter and enable ZIP64
with pd.ExcelWriter(output_file_path, engine='xlsxwriter') as writer:
    writer.book.use_zip64()

    # Split DataFrame into chunks and write each chunk to a separate sheet
    for i in range(0, len(df), max_rows_per_sheet):
        chunk = df.iloc[i:i + max_rows_per_sheet]
        sheet_name = f'Sheet_{i // max_rows_per_sheet + 1}'
        chunk.to_excel(writer, sheet_name=sheet_name, index=False)

print(f"Data processing complete. The new file has been saved as '{output_file_path}'.")


The output file path is 'NonEnsemble-Enhanced Random Forest Input.xlsx'
Data processing complete. The new file has been saved as 'NonEnsemble-Enhanced Random Forest Input.xlsx'.


In [24]:
print('Done')

Done
